In [3]:
import pandas as pd

random_state = 42

In [19]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv"
rating_df = pd.read_csv(rating_url)
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,3.0
1,1342067,CL0101EN,3.0
2,1990814,ML0120ENv3,3.0
3,380098,BD0211EN,3.0
4,779563,DS0101EN,3.0


In [20]:
num_users = len(rating_df['user'].unique())
num_items = len(rating_df['item'].unique())
print(f"There are total {num_users} users and {num_items} items.")

There are total 33901 users and 126 items.


In [21]:
def process_dataset(raw_data):
    encoded_data = raw_data.copy()
    user_list = encoded_data['user'].unique().tolist()
    user_id2idx_dict = {x: i for i, x in enumerate(user_list)}
    user_idx2id_dict = dict(enumerate(user_list))

    course_list = encoded_data['item'].unique().tolist()
    course_id2idx_dict = {x: i for i, x in enumerate(course_list)}
    course_idx2id_dict = dict(enumerate(course_list))

    encoded_data['user'] = encoded_data['user'].map(user_id2idx_dict)
    encoded_data['item'] = encoded_data['item'].map(course_id2idx_dict)
    encoded_data['rating'] = encoded_data['rating'].astype(int)

    return encoded_data, user_idx2id_dict, course_idx2id_dict

In [22]:
encoded_data, user_idx2id_dict, course_idx2id_dict = process_dataset(rating_df)

In [23]:
encoded_data.head()

,user,item,rating
0,0,0,3
1,1,1,3
2,2,2,3
3,3,3,3
4,4,4,3


In [24]:
def generate_train_test_datasets(dataset, scale=True):
    min_rating = min(dataset['rating'])
    max_rating = max(dataset['rating'])

    dataset = dataset.sample(frac=1, random_state=random_state)
    x = dataset[['user', 'item']].values
    if scale:
        y = dataset['rating'].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values
    else:
        y = dataset['rating'].values

    train_indices = int(0.8 * dataset.shape[0])
    test_indices = int(0.9 * dataset.shape[0])

    x_train, x_val, x_test, y_train, y_val, y_test = (
        x[:train_indices],
        x[train_indices:test_indices],
        x[test_indices:],
        y[:train_indices],
        y[train_indices:test_indices],
        y[test_indices:]
    )
    return x_train, x_val, x_test, y_train, y_val, y_test

In [25]:
x_train, x_val, x_test, y_train, y_val, y_test = generate_train_test_datasets(encoded_data)

In [26]:
user_indices = x_train[:, 0]
user_indices

array([ 8376,  7659, 10717, ...,  3409, 28761,  4973], shape=(186644,))

In [27]:
item_indices = x_train[:, 0]
item_indices

array([ 8376,  7659, 10717, ...,  3409, 28761,  4973], shape=(186644,))

In [28]:
y_train

array([1., 1., 1., ..., 1., 0., 1.], shape=(186644,))

In [30]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x_train_tensor = torch.tensor(x_train, dtype=torch.long)
x_val_tensor = torch.tensor(x_val, dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.float)
y_val_tensor = torch.tensor(y_val, dtype=torch.float)
y_test_tensor = torch.tensor(y_test, dtype=torch.float)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

In [32]:
from torch import nn


class RecommenderNet(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)

        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.kaiming_normal_(self.user_embedding.weight)
        nn.init.kaiming_normal_(self.item_embedding.weight)

    def forward(self, inputs):
        users, items = inputs[:, 0], inputs[:, 1]
        user_vector = self.user_embedding(users)
        item_vector = self.item_embedding(items)

        user_bias = self.user_bias(users)
        item_bias = self.item_bias(items)

        dot_user_item = (user_vector * item_vector).sum(dim=1, keepdims=True)
        x = dot_user_item + user_bias + item_bias
        return torch.sigmoid(x).squeeze(1)

In [33]:
model = RecommenderNet(num_users, num_items)

In [36]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
criterion = nn.MSELoss()

In [39]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(train_loader.dataset)

In [40]:
def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(val_loader.dataset)

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
n_epochs = 10
for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 1, Train Loss: 0.3087, Val Loss: 0.2893
Epoch 2, Train Loss: 0.2711, Val Loss: 0.2570
Epoch 3, Train Loss: 0.2359, Val Loss: 0.2266
Epoch 4, Train Loss: 0.2026, Val Loss: 0.1977
Epoch 5, Train Loss: 0.1708, Val Loss: 0.1698
Epoch 6, Train Loss: 0.1401, Val Loss: 0.1427
Epoch 7, Train Loss: 0.1109, Val Loss: 0.1168
Epoch 8, Train Loss: 0.0846, Val Loss: 0.0935
Epoch 9, Train Loss: 0.0625, Val Loss: 0.0740
Epoch 10, Train Loss: 0.0455, Val Loss: 0.0588


In [47]:
test_loss = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.0609


In [48]:
import numpy as np

rmse = np.sqrt(test_loss)
print(f"RMSE: {rmse:.4f}")

RMSE: 0.2467
